# DPNet Example

This notebook loads a processed DPNet task and runs the package-native Random Forest baseline on Morgan fingerprints.

In [1]:
from pathlib import Path
import json
import sys

project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from dpnet import DPNet, BaselineRunConfig, run_baseline

task = "bbbp"
task_dir = project_root / "database" / "distribution" / task / "processed" / task
output_dir = project_root / "tmp" / "example-runs"

task_dir

PosixPath('/home/silong/paper/molm/dpnet/database/distribution/bbbp/processed/bbbp')

## Load Processed Data

In [2]:
dpnet = DPNet(task_dir)

split_sizes = {name: len(dataset) for name, dataset in dpnet.datasets.items()}
split_sizes

{'train': 1466, 'valid': 252, 'test': 160}

In [3]:
dpnet.task_meta.to_dict()

{'name': 'bbbp',
 'version': 1,
 'dialect': 'dpnet',
 'processed_dir': 'processed',
 'id_col': None,
 'smiles_col': 'smiles',
 'strict_test': True,
 'labels': [{'id': 'bbbp',
   'label_col': 'label',
   'problem_type': 'binary',
   'num_classes': 2}],
 'seed': 42,
 'extra_cols': []}

In [4]:
dpnet.datasets["train"][0]

{'cid': 'cid_270',
 'smiles': 'CC/C(=C(\\c1ccccc1)c1ccc(OCCN(C)C)cc1)c1ccccc1',
 'bbbp': 1}

## Run Random Forest Baseline

In [5]:
result = run_baseline(
    BaselineRunConfig(
        task=task,
        model="rf",
        task_dir=task_dir,
        output_dir=output_dir,
        n_estimators=25,
        n_jobs=1,
    )
)

result

[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not r

[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
[20:25:24] WARNING: not removing hydrogen atom without neighbors
2026-06-02 20:25:24.841 |

BaselineRunResult(run_dir=PosixPath('/home/silong/paper/molm/dpnet/tmp/example-runs/bbbp/rf'), metrics_path=PosixPath('/home/silong/paper/molm/dpnet/tmp/example-runs/bbbp/rf/metrics.json'), config_path=PosixPath('/home/silong/paper/molm/dpnet/tmp/example-runs/bbbp/rf/config.json'))

In [6]:
metrics = json.loads(result.metrics_path.read_text())
metrics

{'task': 'bbbp',
 'model': 'rf',
 'seed': 42,
 'features': {'radius': 2, 'n_bits': 2048, 'include_chirality': True},
 'labels': {'bbbp': {'problem_type': 'binary',
   'splits': {'train': {'accuracy': 0.9993178717598908,
     'f1_macro': 0.9991002353749889,
     'roc_auc': 1.0},
    'valid': {'accuracy': 0.9087301587301587,
     'f1_macro': 0.8653814887934038,
     'roc_auc': 0.9490451388888889},
    'test': {'accuracy': 0.8625,
     'f1_macro': 0.8028673835125448,
     'roc_auc': 0.9131994261119083}}}}}

In [7]:
import pandas as pd

test_predictions = pd.read_csv(result.run_dir / "predictions" / "test.csv")
test_predictions.head()

,cid,smiles,bbbp,pred_bbbp,prob_bbbp_0,prob_bbbp_1
0,cid_1117,O=C(CC1c2ccccc2C(=O)N1c1ccc2ccc(Cl)nc2n1)N1CCC...,1,1,0.08,0.92
1,cid_28,CN1[C@@H]2CCC[C@H]1CC(NC(=O)c1nn(C)c3ccccc13)C2,1,1,0.16,0.84
2,cid_325,CC(C)C1NC(=O)[C@@H](C(C)C)OC(=O)[C@H](C(C)C)NC...,0,1,0.32,0.68
3,cid_1071,CCCCCCCCCCCCCC(=O)O[C@H]1C=C[C@H]2[C@H]3Cc4ccc...,1,1,0.16,0.84
4,cid_705,C=CC[C@@H]1/C=C(\C)C[C@H](C)C[C@H](OC)[C@H]2O[...,0,1,0.44,0.56


In [8]:
test_predictions.columns.tolist()

['cid', 'smiles', 'bbbp', 'pred_bbbp', 'prob_bbbp_0', 'prob_bbbp_1']

## Perimeter Split Demo

In [ ]:
from dpnet.preprocess import PERIMETER_MAX_SAMPLES, perimeter_split_df

processed_frames = [
    pd.read_csv(task_dir / f"{split}.csv").assign(source_split=split)
    for split in ["train", "valid", "test"]
]
perimeter_demo_df = pd.concat(processed_frames, ignore_index=True).head(200)

perimeter_splits = perimeter_split_df(
    perimeter_demo_df,
    smiles_col="smiles",
    stratify_col=task,
    random_state=42,
    max_samples=PERIMETER_MAX_SAMPLES,
)

{name: len(frame) for name, frame in perimeter_splits.items()}